<a href="https://colab.research.google.com/github/biglalo104/Projects/blob/main/FP%20Growth%20algorithm%20using%20Apache%20Spark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pyspark.sql import SparkSession
from pyspark.ml.fpm import FPGrowth
import random

# 1. Initialize Spark Session (Local mode for demonstration, use YARN/K8s for production)
spark = SparkSession.builder \
    .appName("Supermarket_Market_Basket_Analysis") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

# 2. Simulate 2 Million Retail Transactions using Distributed RDDs
# We define realistic supermarket categories and inject strong correlations to mimic real shopping behavior.
def generate_transaction(idx):
    categories = {
        "Breakfast": ["Cereal", "Milk", "Coffee", "Bread", "Butter"],
        "Dinner": ["Chicken", "Beef", "Rice", "Pasta", "Tomato_Sauce", "Onions"],
        "Snacks": ["Chips", "Soda", "Beer", "Chocolate"],
        "Baby": ["Diapers", "Wipes", "Baby_Food"],
        "Produce": ["Apples", "Bananas", "Potatoes", "Tomatoes"]
    }

    # 60% chance of a themed basket, 40% random assortment
    if random.random() < 0.6:
        theme = random.choice(list(categories.keys()))
        items = random.sample(categories[theme], random.randint(2, len(categories[theme])))
    else:
        all_items = [item for sublist in categories.values() for item in sublist]
        items = random.sample(all_items, random.randint(2, 4))

    # Inject high-confidence real-world correlations
    if "Diapers" in items and random.random() < 0.8: items.append("Beer")
    if "Pasta" in items and random.random() < 0.85: items.append("Tomato_Sauce")
    if "Bread" in items and random.random() < 0.9: items.append("Butter")
    if "Milk" in items and random.random() < 0.7: items.append("Cereal")

    return (idx, list(set(items)))

print("Generating 2,000,000 simulated transactions...")
num_records = 2000000
# Parallelize across 20 partitions for distributed processing
rdd = spark.sparkContext.parallelize(range(num_records), 20).map(generate_transaction)
df = spark.createDataFrame(rdd, ["transaction_id", "items"])

# 3. Apply FP-Growth Algorithm
# minSupport=0.05 (Itemset must appear in at least 5% of transactions)
# minConfidence=0.60 (Rule must be correct at least 60% of the time)
fpGrowth = FPGrowth(itemsCol="items", minSupport=0.05, minConfidence=0.60)
model = fpGrowth.fit(df)

# 4. Extract and Display Results
print("\n--- TOP FREQUENT ITEMSETS ---")
model.freqItemsets.orderBy("freq", ascending=False).show(15, truncate=False)

print("\n--- SIGNIFICANT ASSOCIATION RULES ---")
# Lift > 1 indicates a positive correlation (purchased together more often than by chance)
rules_df = model.associationRules.orderBy("lift", ascending=False)
rules_df.show(15, truncate=False)

# Optional: Save results to Parquet for downstream BI tools
# model.freqItemsets.write.parquet("s3a://my-data-lake/frequent_itemsets/")
# rules_df.write.parquet("s3a://my-data-lake/association_rules/")

spark.stop()

Generating 2,000,000 simulated transactions...

--- TOP FREQUENT ITEMSETS ---
+--------------+------+
|items         |freq  |
+--------------+------+
|[Beer]        |526724|
|[Butter]      |407269|
|[Tomato_Sauce]|392273|
|[Cereal]      |378115|
|[Wipes]       |309328|
|[Baby_Food]   |309176|
|[Diapers]     |308928|
|[Tomatoes]    |290436|
|[Apples]      |290100|
|[Potatoes]    |289966|
|[Bananas]     |289674|
|[Chips]       |289094|
|[Soda]        |288838|
|[Chocolate]   |288759|
|[Coffee]      |277355|
+--------------+------+
only showing top 15 rows

--- SIGNIFICANT ASSOCIATION RULES ---
+------------------------+----------+------------------+------------------+---------+
|antecedent              |consequent|confidence        |lift              |support  |
+------------------------+----------+------------------+------------------+---------+
|[Baby_Food, Beer]       |[Diapers] |0.9283425279896155|6.019169479479582 |0.0686565|
|[Wipes, Beer]           |[Diapers] |0.9270638200028404|6.